# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmerSajid842/flyrankmlproject/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


```python
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path.cwd() / "data/raw/content_refresh_anonymized.csv",
    Path.cwd().parent / "data/raw/content_refresh_anonymized.csv",
    Path.cwd().parent.parent / "data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Starter data not found in the expected repo locations.")

df = pd.read_csv(data_path)
df[["ctr", "avg_position", "trend_direction", "trend_pct"]].head()

```

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


```python
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["client_id"]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
feature_cols = ["ctr", "avg_position", "engagement_rate", "scroll_rate", "content_age_days", "word_count"]
y_train = train_df["trend_direction"].eq("down")
y_test = test_df["trend_direction"].eq("down")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(train_df[feature_cols].fillna(0), y_train)
scores = model.predict_proba(test_df[feature_cols].fillna(0))[:, 1]
scores[:5]

```

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


```python
roc_auc_score(y_test, scores)

```

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


```python
top_candidates = test_df.iloc[scores.argsort()[::-1], :][["content_id", "trend_direction", "ctr", "avg_position"]].head(20)
top_candidates

```

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmerSajid842/flyrankmlproject/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 – from page 6 of the paper
*"Growing content is 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days)."*

**Methodology question:**  
The label "growing" is defined as a 30‑day impression change >10%. Was this label computed independently of the features used for comparison (word count, age)? If the same time window was used to calculate both the label and the feature averages, then the comparison might partly reflect co‑occurrence rather than a predictive relationship.

---

### Finding 2 – from page 7 of the paper
*"Content peaks at 61‑90 days, then declines after 270 days; refreshed older pages can recover."*

**Methodology question:**  
The health score is a composite of impressions, position, CTR, and scroll depth. Does this composite use the same 90‑day window as the age‑based grouping? If so, the pattern may be descriptive of the current snapshot rather than a causal age effect. I would be interested in seeing an age‑controlled refresh analysis that separates initial publication from later updates.

### Uploading Files

Use the following cell to upload any necessary files to your Colab environment. After running the cell, a 'Choose Files' button will appear, allowing you to select and upload files from your local machine.

In [2]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
User uploaded file "content_refresh_anonymized.csv" with length 6727670 bytes


In [4]:
from pathlib import Path
import pandas as pd

# Assuming the file was uploaded directly to the Colab environment's root directory
data_path = Path("/content/content_refresh_anonymized.csv")

if not data_path.exists():
    raise FileNotFoundError(f"File not found at {data_path}. Please ensure 'content_refresh_anonymized.csv' was uploaded correctly.")

df = pd.read_csv(data_path)
df[["ctr", "avg_position", "trend_direction", "trend_pct"]].head()

,ctr,avg_position,trend_direction,trend_pct
0,0.76,10.6,down,-41.4
1,0.05,20.3,down,-57.7
2,0.09,36.5,down,-60.9
3,0.49,6.2,stable,-13.8
4,0.13,44.0,down,-34.7


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["client_id"]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
feature_cols = ["ctr", "avg_position", "engagement_rate", "scroll_rate", "content_age_days", "word_count"]
y_train = train_df["trend_direction"].eq("down")
y_test = test_df["trend_direction"].eq("down")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(train_df[feature_cols].fillna(0), y_train)
scores = model.predict_proba(test_df[feature_cols].fillna(0))[:, 1]
scores[:5]


array([0.33, 0.38, 0.73, 0.8 , 0.61])

In [6]:
# ---- Week‑5 style (random split) for comparison ----
from sklearn.model_selection import train_test_split

X = df[feature_cols].fillna(0)
y = df["trend_direction"].eq("down")

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_rand = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_rand.fit(X_train_rand, y_train_rand)
scores_rand = model_rand.predict_proba(X_test_rand)[:, 1]
auc_rand = roc_auc_score(y_test_rand, scores_rand)

# Grouped split AUC (you already computed it above)
auc_grouped = roc_auc_score(y_test, scores)   # from your earlier cell

print(f"Random split AUC (Week‑5 style): {auc_rand:.4f}")
print(f"Grouped split AUC (this audit):     {auc_grouped:.4f}")

Random split AUC (Week‑5 style): 0.7148
Grouped split AUC (this audit):     0.5369


### Before vs After – Validation Performance

| Validation Method | AUC |
|-------------------|-----|
| Random split (Week 5) | 0.7148|
| Grouped split (this audit) | 0.5369 |

**Interpretation:**  
The grouped split gave a **lower** AUC than the random split. This suggests that the random split may have **over‑estimated** performance because pages from the same client could appear in both training and test sets. The grouped split is more conservative and better reflects how the model would perform on new clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
### Feature‑by‑Feature Leakage Check

| Feature | Available at decision time? | Risk of leakage | Decision |
|---------|-----------------------------|-----------------|----------|
| ctr | Yes – historical CTR | Low | Keep |
| avg_position | Yes – historical position | Low | Keep |
| engagement_rate | Yes – historical engagement | Low | Keep |
| scroll_rate | Yes – historical scroll | Low | Keep |
| content_age_days | Yes – as of prediction date | Low | Keep |
| word_count | Yes – static page attribute | Low | Keep |

All features are computed from past or static data. The target (`trend_direction`) is based on a future 30‑day change relative to the previous 30 days, but it is **not** used as a feature. Therefore, no serious leakage is identified in this feature set.

In [7]:
roc_auc_score(y_test, scores)


np.float64(0.536919905688348)

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (bold) claim from my earlier work:**  
*"My model reliably identifies which pages are going to decline, so you should prioritize refreshing them."*

**Rewritten (safe, decision‑support version):**  
*"In this dataset, the model showed an observed ability to rank pages by their likelihood of being associated with a future decline (based on the defined trend label). This can be used as a directional signal to prioritise pages for manual review, but it does not establish that refreshing those pages will cause recovery – the model only points to correlation, not causation."*

In [8]:
top_candidates = test_df.iloc[scores.argsort()[::-1], :][["content_id", "trend_direction", "ctr", "avg_position"]].head(20)
top_candidates


,content_id,trend_direction,ctr,avg_position
15105,content_d089619f78d7,flat,0.00,5.5
3771,content_d6dd255e10bc,down,0.00,16.1
29663,content_1980254ef582,down,0.00,13.9
1063,content_34ad541ab004,down,0.31,15.7
12402,content_2276d67dccae,flat,0.00,7.3
15368,content_0065d16af779,flat,0.00,8.9
9549,content_be905335c59c,flat,0.00,7.8
28596,content_e0f75f03ddb9,down,0.00,7.5
5918,content_437d6e67bfe2,down,0.00,15.7
2200,content_dc0eaf635a88,flat,0.00,8.9


### 4b. Error analysis – examples of misclassifications

In [9]:
# Get predictions on test set
y_pred = (scores >= 0.5).astype(int)   # or use model.predict(...)

# Create a copy of test data with predictions and actuals
test_eval = test_df.copy()
test_eval['actual'] = y_test.astype(int)
test_eval['predicted'] = y_pred
test_eval['score'] = scores

# False positives: predicted decline, but actual not
fp = test_eval[(test_eval['predicted'] == 1) & (test_eval['actual'] == 0)]
# False negatives: predicted not decline, but actual decline
fn = test_eval[(test_eval['predicted'] == 0) & (test_eval['actual'] == 1)]

print(f"False positives: {len(fp)}")
print(f"False negatives: {len(fn)}")

# Show a few examples
print("\nTop 5 false positives (high confidence wrong):")
fp_sorted = fp.sort_values('score', ascending=False)
display(fp_sorted[['content_id', 'ctr', 'avg_position', 'engagement_rate', 'content_age_days', 'actual', 'predicted', 'score']].head(5))

print("\nTop 5 false negatives (high confidence missed):")
fn_sorted = fn.sort_values('score', ascending=False)  # highest probability of decline but predicted safe
display(fn_sorted[['content_id', 'ctr', 'avg_position', 'engagement_rate', 'content_age_days', 'actual', 'predicted', 'score']].head(5))

False positives: 1700
False negatives: 1198

Top 5 false positives (high confidence wrong):


,content_id,ctr,avg_position,engagement_rate,content_age_days,actual,predicted,score
15105,content_d089619f78d7,0.0,5.5,0.0,223,0,1,1.00
12402,content_2276d67dccae,0.0,7.3,0.0,460,0,1,0.98
9549,content_be905335c59c,0.0,7.8,0.0,223,0,1,0.98
2200,content_dc0eaf635a88,0.0,8.9,0.0,132,0,1,0.98
15368,content_0065d16af779,0.0,8.9,0.0,460,0,1,0.98



Top 5 false negatives (high confidence missed):


,content_id,ctr,avg_position,engagement_rate,content_age_days,actual,predicted,score
13082,content_cbd542e4e064,0.02,9.0,0.0,421,1,0,0.496667
28503,content_8d7c1d704b43,0.00,3.1,0.0,181,1,0,0.490000
13364,content_451118cef0ac,0.24,21.6,0.0,230,1,0,0.490000
13588,content_e8743787bbef,0.02,7.5,0.0,487,1,0,0.490000
1251,content_1aef96f28d03,0.00,5.7,0.0,275,1,0,0.490000


“Many false negatives occurred for pages with low historical engagement, suggesting the model may be too cautious when signals are sparse. False positives often involved pages with poor positions but high CTR – possibly an over‑reliance on CTR as a proxy.”

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.